# 基于 CANN 的 Softmax 算子实现实验

Softmax 是深度学习中连接线性层和概率分布的基础算子，也是一个很典型的“按行归约 + 逐元素归一化”问题。它不像 Add 那样只依赖当前位置：一行中的最大值和指数和会影响这一行的每一个输出，因此实现时必须同时考虑数值稳定性、连续访存、片上缓冲和多核任务划分。

本实验在 Ascend C 编程模型下实现二维 `float32` 张量最后一维上的稳定 Softmax，并通过 ACL 直调验证 Kernel。实验特别强调编译边界：`softmax.asc` 只保留设备侧 Kernel 和 ASC 启动包装函数；`host.cpp` 独立承担 Tiling、ACL 资源管理、CPU 参考实现和结果统计。

学习大纲如下：

1. 环境准备：创建工程目录，检查 CANN 环境变量和 NPU 条件；
2. 问题分析：理解稳定 Softmax 的数学形式、二维张量布局、行级并行策略，以及原单文件工程的编译失败原因；
3. 数据结构与 Kernel：认识 `GlobalTensor`、`LocalTensor`、`TQue`、`TBuf` 和 `SoftMaxTiling` 在不同存储层次中的职责；
4. Host 验证：生成可重复输入，计算 CPU golden，完成 Tiling、内存管理、Kernel 发射、计时和误差校验；
5. 构建与结果分析：使用 ASC/CXX 分离编译，读取三组规模的准确率、时延和吞吐量；
6. 实验总结与拓展：从访存连续性、归约开销和并行度角度解释现象，并设计后续变量控制实验。

运行最后的编译和运行单元后，程序会执行 `16x1024`、`32x1024` 和 `64x1024` 三组测试，输出正确性指标、平均时延、吞吐量，并将汇总表保存到 `Source/04.00/softmax/softmax_results.txt`。

## 1. 环境准备

下面创建工程目录并查找、加载 CANN 根目录下的 `set_env.sh`。编译和运行需要 Linux、CANN 8.3.RC1（或兼容版本）以及可用的昇腾 NPU。没有 NPU 时仍可先执行代码生成单元，检查工程文件和注释是否完整。

生成后的目录结构如下：

```text
Source/04.00/softmax/
├── softmax_common.h   # Host 与 Kernel 共享的轻量配置
├── softmax.asc         # ASC 编译的设备侧 Kernel
├── host.cpp            # CXX 编译的 Host、Tiling 和校验
├── CMakeLists.txt      # ASC/CXX 混合目标和链接配置
├── build/              # CMake 构建目录
└── softmax_results.txt # 运行输出和汇总表
```

环境脚本由本单元先加载到 Notebook 的 Python 进程，再由编译和运行单元传递给 Bash。若脚本不存在，Notebook 会明确提示，而不会把一个缺少 CANN 依赖的半成品误认为编译成功。

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

project_dir = Path('Source/04.00/softmax')
project_dir.mkdir(parents=True, exist_ok=True)

# 与同目录 Ascend C 实验保持一致：优先使用已有变量，再扫描常见安装目录。
candidate_roots = [
    os.environ.get('ASCEND_TOOLKIT_HOME'),
    os.environ.get('ASCEND_INSTALL_PATH'),
    os.environ.get('ASCEND_HOME_PATH'),
    os.environ.get('ASCEND_CANN_PACKAGE_PATH'),
    '/usr/local/Ascend/ascend-toolkit/latest',
    '/usr/local/Ascend/ascend-toolkit',
    '/opt/Ascend/ascend-toolkit/latest',
    '/opt/Ascend/ascend-toolkit',
    '/opt/conda/Ascend/cann-9.0.0/aarch64-linux',
    str(Path.home() / 'Ascend/ascend-toolkit/latest'),
    str(Path.home() / 'Ascend/ascend-toolkit'),
    '/workspace/Ascend/ascend-toolkit/latest',
    '/workspace/Ascend/ascend-toolkit',
]
set_env_path = None
for root in candidate_roots:
    if root:
        candidate = Path(root) / 'set_env.sh'
        if candidate.exists():
            set_env_path = candidate
            break

if set_env_path is None:
    for root in [Path('/usr/local/Ascend'), Path('/opt/Ascend'), Path('/opt/conda/Ascend'), Path.home(), Path('/workspace')]:
        if root.exists():
            matches = list(root.glob('**/ascend-toolkit*/set_env.sh')) + list(root.glob('**/cann-*/**/set_env.sh'))
            if matches:
                set_env_path = matches[0]
                break

if set_env_path is not None:
    env_text = subprocess.check_output(
        ['bash', '-lc', f'source {shlex.quote(str(set_env_path))} && env'],
        text=True,
    )
    for line in env_text.splitlines():
        if '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value
    # 某些版本只导出 ASCEND_HOME_PATH，统一补齐 CMake 使用的变量。
    os.environ.setdefault('ASCEND_INSTALL_PATH', str(set_env_path.parent))
    os.environ.setdefault('ASCEND_CANN_PACKAGE_PATH', os.environ['ASCEND_INSTALL_PATH'])
    print('Ascend environment loaded from:', set_env_path)
else:
    print('Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.')

print('Experiment directory:', project_dir.resolve())

## 2. 问题分析与编译域设计

### 2.1 稳定 Softmax 的数学形式

对一行输入 $x=[x_1,\ldots,x_n]$，Softmax 定义为：

$$\operatorname{softmax}(x_i)=\frac{e^{x_i}}{\sum_j e^{x_j}}$$

直接计算指数可能溢出或下溢。本实验先求行最大值 $m=\max_j x_j$，再计算：

$$y_i=\frac{e^{x_i-m}}{\sum_j e^{x_j-m}}$$

减去最大值不改变最终比例，却把指数输入限制在不大于 0 的范围内。Kernel 使用高阶 `AscendC::SoftMax` 接口完成这组归约和归一化。

### 2.2 张量布局和并行映射

输入和输出是连续的二维 ND 张量 `[rows, cols]`，元素 `(r,c)` 的线性位置为 `r * cols + c`。一行连续存储，适合一次 `DataCopy` 搬入 Local Memory；不同行之间没有数据依赖，适合按 AI Core 循环分配：第 `blockIdx` 个核处理 `blockIdx`、`blockIdx + blockNum` 等行。行内的最大值和求和仍然是归约依赖，不能简单地把每个元素当成完全独立的任务。

### 2.3 原始失败现象和根因

原版本把 `main`、`CpuSoftmax`、`RunKernel`、`BuildSoftmaxKernelConfig` 等 Host 代码和 `KernelSoftmax`、`softmax_custom` 放在同一个 `.asc` 文件中。ASC 编译器会把整个翻译单元放入 Kernel 编译和链接流程，因此 Host 侧的 `std::exp`/`logl`、GE 的 `ge::Shape`、Tiling 注册符号以及 ACL 运行时符号无法在该链接域中解析，最终在编译单元中触发多个 `undefined symbol`。

修复原则不是屏蔽某几个符号，而是按职责拆分编译域：ASC 文件只编译设备代码；CXX 文件编译标准库、GE/Tiling 和 ACL Host 代码。ASC 文件额外导出 `softmax_custom_do`，它在 ASC 编译域内部使用 `<<<>>>` 启动语法，CXX Host 只调用这个标准 C 接口。

### 2.4 共享数据结构

Host 与 Kernel 通过 `SoftmaxKernelConfig` 传递行数、行长、临时空间大小和 `SoftMaxTiling`。`GlobalTensor` 表示 GM 中的整块输入/输出，`LocalTensor` 表示当前行的片上数据，`TQue` 负责输入输出 Tensor 的所有权转移，`TBuf` 保存归约结果和 Tiling 要求的临时空间。

Host 与 Kernel 共享的常量和 `SoftmaxKernelConfig` 放在 `softmax_common.h` 中。公共头文件只描述接口数据，不放置 `main`、标准库算法或 GE 构造逻辑。

- `softmax.asc`：`KernelSoftmax`、GM/LM 搬运、`AscendC::SoftMax`、Kernel 入口和 `softmax_custom_do` 启动包装函数。
- `host.cpp`：输入生成、CPU golden、`GetSoftMaxMinTmpSize`/`SoftMaxTilingFunc`、ACL 内存与 Stream、结果校验；只调用 `softmax_custom_do`，不包含 `<<<>>>` 语法。
- `CMakeLists.txt`：`add_executable(softmax_demo softmax.asc host.cpp)`，由文件扩展名选择 ASC/CXX 编译器。

这种边界避免 ASC 编译阶段尝试链接 `logl`、`expf`、`ge::Shape` 和 Tiling 注册符号。

## 3. Kernel 开发与 Host 验证

接下来的代码单元按“共享配置 → ASC Kernel → CXX Host”的顺序生成文件。Kernel 负责一行数据的搬入、Softmax 计算和写回；Host 负责把运行时形状转成 Tiling 数据、准备 ACL 资源并验收结果。这样的顺序既对应程序真实调用链，也方便在出现错误时定位是设备计算、Host 运行时还是链接配置的问题。

In [ ]:
%%writefile Source/04.00/softmax/softmax_common.h
#pragma once

#include <cstdint>
#include "tiling/tiling_api.h"

constexpr uint32_t kBufferNum = 1;
constexpr uint32_t kSumMaxBlockLen = 32 / sizeof(float);
constexpr uint32_t kDefaultLocalWorkspaceSize = 64 * 1024;

struct SoftmaxKernelConfig {
    uint32_t rowNum;
    uint32_t rowLen;
    uint32_t localWorkspaceSize;
    SoftMaxTiling softmaxTilingData;
};


In [ ]:
%%writefile Source/04.00/softmax/softmax.asc
#include "kernel_operator.h"
#include "softmax_common.h"

class KernelSoftmax {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR z, const SoftmaxKernelConfig &config)
    {
        rowNum = config.rowNum;
        rowLen = config.rowLen;
        softmaxTiling = config.softmaxTilingData;
        xGm.SetGlobalBuffer((__gm__ float *)x, rowNum * rowLen);
        zGm.SetGlobalBuffer((__gm__ float *)z, rowNum * rowLen);
        pipe.InitBuffer(inQueueX, kBufferNum, rowLen * sizeof(float));
        pipe.InitBuffer(outQueueZ, kBufferNum, rowLen * sizeof(float));
        pipe.InitBuffer(sumBuf, kSumMaxBlockLen * sizeof(float));
        pipe.InitBuffer(maxBuf, kSumMaxBlockLen * sizeof(float));
        pipe.InitBuffer(tmpBuf, config.localWorkspaceSize);
    }

    __aicore__ inline void Process()
    {
        AscendC::SoftMaxShapeInfo info {1U, rowLen, 1U, rowLen};
        for (uint32_t row = AscendC::GetBlockIdx(); row < rowNum; row += AscendC::GetBlockNum()) {
            const uint32_t offset = row * rowLen;
            AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
            AscendC::DataCopy(xLocal, xGm[offset], rowLen);
            inQueueX.EnQue(xLocal);
            xLocal = inQueueX.DeQue<float>();
            AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
            AscendC::LocalTensor<float> sumLocal = sumBuf.Get<float>();
            AscendC::LocalTensor<float> maxLocal = maxBuf.Get<float>();
            AscendC::LocalTensor<uint8_t> tmpLocal = tmpBuf.Get<uint8_t>();
            AscendC::SoftMax<float>(zLocal, sumLocal, maxLocal, xLocal, tmpLocal, softmaxTiling, info);
            outQueueZ.EnQue<float>(zLocal);
            inQueueX.FreeTensor(xLocal);
            zLocal = outQueueZ.DeQue<float>();
            AscendC::DataCopy(zGm[offset], zLocal, rowLen);
            outQueueZ.FreeTensor(zLocal);
        }
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, kBufferNum> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECOUT, kBufferNum> outQueueZ;
    AscendC::TBuf<AscendC::TPosition::VECCALC> sumBuf;
    AscendC::TBuf<AscendC::TPosition::VECCALC> maxBuf;
    AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> zGm;
    SoftMaxTiling softmaxTiling {};
    uint32_t rowNum = 0;
    uint32_t rowLen = 0;
};

extern "C" __global__ __aicore__ void softmax_custom(GM_ADDR x, GM_ADDR z, SoftmaxKernelConfig config)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelSoftmax op;
    op.Init(x, z, config);
    op.Process();
}

// ASC 编译单元导出标准 C 接口；CXX Host 通过它发射 Kernel。
extern "C" void softmax_custom_do(uint32_t blockDim, void *stream, uint8_t *x, uint8_t *z, SoftmaxKernelConfig config)
{
    softmax_custom<<<blockDim, nullptr, stream>>>(x, z, config);
}


In [ ]:
%%writefile Source/04.00/softmax/host.cpp
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <limits>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>
#include "acl/acl.h"
#include "graph/tensor.h"
#include "softmax_common.h"

constexpr uint32_t kDefaultRows = 16;
constexpr uint32_t kDefaultCols = 1024;
constexpr uint32_t kDefaultRepeat = 100;
constexpr uint32_t kDefaultWarmup = 10;
constexpr uint32_t kDefaultBlockDim = 8;
constexpr float kVerifyEps = 1e-4f;
struct RunOptions { uint32_t rows=kDefaultRows, cols=kDefaultCols, repeat=kDefaultRepeat, warmup=kDefaultWarmup, blockDim=kDefaultBlockDim; bool suite=true; };
struct VerifyMetrics { float maxAbs=0, maxRowSum=0; bool passed=false; };
struct Result { uint32_t rows=0, cols=0; double avgMs=0; VerifyMetrics verify; std::vector<float> output; };
#define ACL_CHECK(expr) do { aclError s=(expr); if (s != ACL_SUCCESS) throw std::runtime_error(std::string("ACL call failed: ")+#expr+" ("+std::to_string((int)s)+")"); } while (0)

uint32_t ReadUint(const std::string &value, const char *name) { if (value.empty()) throw std::invalid_argument(std::string("missing ")+name); unsigned long n=std::stoul(value); if (n==0 || n>std::numeric_limits<uint32_t>::max()) throw std::invalid_argument(std::string("invalid ")+name); return (uint32_t)n; }
RunOptions ParseArgs(int argc, char **argv) {
    RunOptions o;
    for (int i=1;i<argc;++i) { std::string a=argv[i]; if (a=="--suite") {o.suite=true; continue;} if (a=="--single") {o.suite=false; continue;}
        auto read=[&](const char *name)->std::string { std::string p=std::string(name)+"="; if (a.rfind(p,0)==0) return a.substr(p.size()); if (a==name && i+1<argc) return argv[++i]; return {}; };
        std::string v; if (!(v=read("--rows")).empty()) {o.rows=ReadUint(v,"--rows");o.suite=false;} else if (!(v=read("--cols")).empty()) {o.cols=ReadUint(v,"--cols");o.suite=false;} else if (!(v=read("--repeat")).empty()) o.repeat=ReadUint(v,"--repeat"); else if (!(v=read("--warmup")).empty()) o.warmup=ReadUint(v,"--warmup"); else if (!(v=read("--blockdim")).empty()) o.blockDim=ReadUint(v,"--blockdim"); else if (a!="--help" && a!="-h") throw std::invalid_argument("unknown argument: "+a);
    } return o;
}
void Validate(const RunOptions &o) { if (!o.rows || !o.cols || !o.repeat || !o.blockDim) throw std::invalid_argument("rows, cols, repeat and blockdim must be positive"); if (o.cols % kSumMaxBlockLen) throw std::invalid_argument("cols must be a multiple of 8 for float32"); }
std::vector<float> MakeInput(uint32_t rows,uint32_t cols) { std::mt19937 rng(20260707); std::uniform_real_distribution<float> d(-3,3); std::vector<float> x((size_t)rows*cols); for(uint32_t r=0;r<rows;++r) for(uint32_t c=0;c<cols;++c) x[(size_t)r*cols+c]=d(rng)+0.01f*r; return x; }
std::vector<float> CpuSoftmax(const std::vector<float> &x,uint32_t rows,uint32_t cols) { std::vector<float> y(x.size()); for(uint32_t r=0;r<rows;++r){size_t off=(size_t)r*cols; float m=*std::max_element(x.begin()+off,x.begin()+off+cols); double sum=0; for(uint32_t c=0;c<cols;++c){y[off+c]=std::exp(x[off+c]-m);sum+=y[off+c];} for(uint32_t c=0;c<cols;++c)y[off+c]=(float)(y[off+c]/sum);} return y; }
uint32_t AlignUp(uint32_t n,uint32_t a){return (n+a-1)/a*a;}
SoftmaxKernelConfig BuildConfig(uint32_t rows,uint32_t cols) { ge::Shape shape({1,(int64_t)cols}); uint32_t minTmp=AscendC::GetSoftMaxMinTmpSize(shape,sizeof(float),false); SoftmaxKernelConfig c{}; c.rowNum=rows;c.rowLen=cols;c.localWorkspaceSize=AlignUp(std::max(minTmp,kDefaultLocalWorkspaceSize),32); AscendC::SoftMaxTilingFunc(shape,sizeof(float),c.localWorkspaceSize,c.softmaxTilingData); return c; }
VerifyMetrics Verify(const std::vector<float> &out,const std::vector<float> &gold,uint32_t rows,uint32_t cols) { VerifyMetrics m; for(size_t i=0;i<out.size();++i)m.maxAbs=std::max(m.maxAbs,std::fabs(out[i]-gold[i])); for(uint32_t r=0;r<rows;++r){double s=0;for(uint32_t c=0;c<cols;++c)s+=out[(size_t)r*cols+c];m.maxRowSum=std::max(m.maxRowSum,(float)std::fabs(s-1));}m.passed=m.maxAbs<=kVerifyEps&&m.maxRowSum<=kVerifyEps;return m; }

extern "C" void softmax_custom_do(uint32_t blockDim, void *stream, uint8_t *x, uint8_t *z, SoftmaxKernelConfig config);
Result Run(const std::vector<float> &input,const std::vector<float> &gold,const RunOptions &o) { Result r; r.rows=o.rows;r.cols=o.cols; size_t bytes=input.size()*sizeof(float); uint32_t blocks=std::min(o.rows,std::max(1u,o.blockDim)); SoftmaxKernelConfig config=BuildConfig(o.rows,o.cols); aclrtStream stream=nullptr; void *xd=nullptr,*zd=nullptr,*zh=nullptr; ACL_CHECK(aclInit(nullptr)); ACL_CHECK(aclrtSetDevice(0)); ACL_CHECK(aclrtCreateStream(&stream)); ACL_CHECK(aclrtMalloc(&xd,bytes,ACL_MEM_MALLOC_HUGE_FIRST)); ACL_CHECK(aclrtMalloc(&zd,bytes,ACL_MEM_MALLOC_HUGE_FIRST)); ACL_CHECK(aclrtMallocHost(&zh,bytes)); ACL_CHECK(aclrtMemcpy(xd,bytes,input.data(),bytes,ACL_MEMCPY_HOST_TO_DEVICE)); auto launch=[&](){softmax_custom_do(blocks, stream, (uint8_t*)xd, (uint8_t*)zd, config);}; for(uint32_t i=0;i<o.warmup;++i)launch(); ACL_CHECK(aclrtSynchronizeStream(stream)); auto t0=std::chrono::steady_clock::now(); for(uint32_t i=0;i<o.repeat;++i)launch(); ACL_CHECK(aclrtSynchronizeStream(stream)); auto t1=std::chrono::steady_clock::now(); r.avgMs=std::chrono::duration<double,std::milli>(t1-t0).count()/o.repeat; ACL_CHECK(aclrtMemcpy(zh,bytes,zd,bytes,ACL_MEMCPY_DEVICE_TO_HOST)); r.output.assign((float*)zh,(float*)zh+input.size()); r.verify=Verify(r.output,gold,o.rows,o.cols); ACL_CHECK(aclrtFree(xd));ACL_CHECK(aclrtFree(zd));ACL_CHECK(aclrtFreeHost(zh));ACL_CHECK(aclrtDestroyStream(stream));ACL_CHECK(aclrtResetDevice(0));ACL_CHECK(aclFinalize()); return r; }
void Print(const Result &r) { double th=(double)r.rows*r.cols/(r.avgMs*1e-3)/1e6; std::cout<<r.rows<<"x"<<r.cols<<" avg_ms="<<std::fixed<<std::setprecision(4)<<r.avgMs<<" throughput(MElts/s)="<<std::setprecision(2)<<th<<" max_abs_error="<<std::setprecision(6)<<r.verify.maxAbs<<" max_row_sum_error="<<r.verify.maxRowSum<<" "<<(r.verify.passed?"PASS":"FAIL")<<'\n'; }
int main(int argc,char **argv) { try { RunOptions base=ParseArgs(argc,argv); std::vector<std::pair<uint32_t,uint32_t>> cases=base.suite?std::vector<std::pair<uint32_t,uint32_t>>{{16,1024},{32,1024},{64,1024}}:std::vector<std::pair<uint32_t,uint32_t>>{{base.rows,base.cols}}; std::cout<<"=== Softmax results ===\nshape\tavg_ms\tthroughput(MElts/s)\tmax_abs_error\tmax_row_sum_error\tstatus\n"; int rc=0; for(auto [rows,cols]:cases){RunOptions o=base;o.rows=rows;o.cols=cols;Validate(o);auto x=MakeInput(rows,cols);auto g=CpuSoftmax(x,rows,cols);auto r=Run(x,g,o); double th=(double)rows*cols/(r.avgMs*1e-3)/1e6; std::cout<<rows<<"x"<<cols<<'\t'<<std::fixed<<std::setprecision(4)<<r.avgMs<<'\t'<<std::setprecision(2)<<th<<'\t'<<std::setprecision(6)<<r.verify.maxAbs<<'\t'<<r.verify.maxRowSum<<'\t'<<(r.verify.passed?"PASS":"FAIL")<<'\n'; if(!r.verify.passed)rc=1;} return rc; } catch(const std::exception &e){std::cerr<<"Error: "<<e.what()<<'\n';return 1;} }


## 4. CMake 构建配置

`softmax.asc` 和 `host.cpp` 必须同时加入同一个可执行目标。CMake 根据扩展名分别使用 ASC、CXX 编译器；Host 目标再链接 ACL 运行时以及生成 Tiling 所需的 `tiling_api`、`register`、`graph_base` 和可选的 `platform`。ASC 编译器还会在 `.asc` 对象中生成 NPU 启动桩，因此需要同时链接 `ascendc_runtime`、`runtime`、`profapi` 等运行库。`tiling_api` 在 CANN 9.0 中还会调用日志接口，因此必须在它后面显式链接提供 `CheckLogLevel` 的 `unified_dlog`，否则 GNU ld 会报告 `DSO missing from command line`。

这里的 `ASCEND_ROOT` 同时承担三项职责：定位 Ascend C 的 CMake 模块、查找 Host 头文件、查找 Host 链接库。配置阶段若提示库缺失，应先检查 `set_env.sh` 是否加载，以及 `ASCEND_INSTALL_PATH`/`ASCEND_HOME_PATH` 是否指向 CANN Toolkit 根目录。

In [ ]:
%%writefile Source/04.00/softmax/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
if(DEFINED ENV{ASCEND_INSTALL_PATH} AND NOT "$ENV{ASCEND_INSTALL_PATH}" STREQUAL "")
  set(ASCEND_ROOT "$ENV{ASCEND_INSTALL_PATH}")
elseif(DEFINED ENV{ASCEND_HOME_PATH} AND NOT "$ENV{ASCEND_HOME_PATH}" STREQUAL "")
  set(ASCEND_ROOT "$ENV{ASCEND_HOME_PATH}")
endif()
if(ASCEND_ROOT)
  list(PREPEND CMAKE_PREFIX_PATH
    "${ASCEND_ROOT}/tools/tikcpp/ascendc_kernel_cmake"
    "${ASCEND_ROOT}/compiler/tikcpp/ascendc_kernel_cmake"
    "${ASCEND_ROOT}/tikcpp/ascendc_kernel_cmake"
  )
endif()
find_package(ASC REQUIRED)
project(softmax_sample LANGUAGES ASC CXX)
add_executable(softmax_demo softmax.asc host.cpp)
target_compile_options(softmax_demo PRIVATE $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>)
if(ASCEND_ROOT)
  set(CANN_LIB_DIRS
    "${ASCEND_ROOT}/lib64"
    "${ASCEND_ROOT}/compiler/lib64"
    "${ASCEND_ROOT}/runtime/lib64"
    "${ASCEND_ROOT}/runtime/lib64/stub"
  )
  find_library(TILING_API_LIB tiling_api PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(REGISTER_LIB register PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(GRAPH_BASE_LIB graph_base PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(PLATFORM_LIB platform PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(UNIFIED_DLOG_LIB unified_dlog PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(ASCENDC_RUNTIME_LIB ascendc_runtime PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(RUNTIME_LIB runtime PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(ERROR_MANAGER_LIB error_manager PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(PROFAPI_LIB profapi PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(GE_COMMON_BASE_LIB ge_common_base PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(MMPA_LIB mmpa PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(ASCEND_DUMP_LIB ascend_dump PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  find_library(C_SEC_LIB c_sec PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  target_include_directories(softmax_demo PRIVATE "${ASCEND_ROOT}/include" "${ASCEND_ROOT}/compiler/include" "${ASCEND_ROOT}/runtime/include")
  find_library(ASCENDCL_LIB ascendcl PATHS ${CANN_LIB_DIRS} NO_DEFAULT_PATH)
  if(NOT ASCENDCL_LIB OR NOT TILING_API_LIB OR NOT REGISTER_LIB OR NOT GRAPH_BASE_LIB OR NOT PLATFORM_LIB OR NOT UNIFIED_DLOG_LIB OR NOT ASCENDC_RUNTIME_LIB OR NOT RUNTIME_LIB OR NOT ERROR_MANAGER_LIB OR NOT PROFAPI_LIB OR NOT GE_COMMON_BASE_LIB OR NOT MMPA_LIB OR NOT ASCEND_DUMP_LIB OR NOT C_SEC_LIB)
    message(FATAL_ERROR "CANN host/runtime libraries not found; check ASCEND_INSTALL_PATH/ASCEND_HOME_PATH")
  endif()
  # Static tiling_api must precede the shared libraries that satisfy its symbols.
  target_link_libraries(softmax_demo PRIVATE
    ${TILING_API_LIB}
    ${REGISTER_LIB}
    ${GRAPH_BASE_LIB}
    ${PLATFORM_LIB}
    ${ASCENDC_RUNTIME_LIB}
    ${RUNTIME_LIB}
    ${ASCENDCL_LIB}
    ${ERROR_MANAGER_LIB}
    ${PROFAPI_LIB}
    ${GE_COMMON_BASE_LIB}
    ${UNIFIED_DLOG_LIB}
    ${MMPA_LIB}
    ${ASCEND_DUMP_LIB}
    ${C_SEC_LIB}
    m
    dl
  )
endif()


## 5. 编译、运行和结果解析

以下单元会加载 CANN 环境、配置并编译工程，然后运行 `softmax_demo`。运行输出同时写入结果文件；如果任意一组的误差超过 `1e-4`，程序返回非零状态，便于 Notebook 发现失败。

一次完整运行的调用链是：Host 生成输入 → CPU 计算 golden → Host 生成 `SoftMaxTiling` → ACL 申请和拷贝内存 → `softmax_custom_do` 发射 ASC Kernel → Stream 同步 → 结果回传 → 比较最大绝对误差和每行和误差。这个顺序也对应实际工程中从数据准备到算子验收的典型流程。

In [ ]:
%%bash
set -e
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_HOME_PATH:-/usr/local/Ascend/ascend-toolkit/latest}}"
if [ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]; then
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
elif [ -f "${ASCEND_INSTALL_PATH}/bin/setenv.bash" ]; then
  source "${ASCEND_INSTALL_PATH}/bin/setenv.bash"
else
  echo "未找到 CANN 环境脚本：${ASCEND_INSTALL_PATH}/set_env.sh" >&2
  exit 1
fi
export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CANN_PACKAGE_PATH:-${ASCEND_INSTALL_PATH}}"
cmake -S Source/04.00/softmax -B Source/04.00/softmax/build
cmake --build Source/04.00/softmax/build -j


In [ ]:
%%bash
set -o pipefail
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_HOME_PATH:-/usr/local/Ascend/ascend-toolkit/latest}}"
if [ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]; then
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
elif [ -f "${ASCEND_INSTALL_PATH}/bin/setenv.bash" ]; then
  source "${ASCEND_INSTALL_PATH}/bin/setenv.bash"
else
  echo "未找到 CANN 环境脚本：${ASCEND_INSTALL_PATH}/set_env.sh" >&2
  exit 1
fi
export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CANN_PACKAGE_PATH:-${ASCEND_INSTALL_PATH}}"
./Source/04.00/softmax/build/softmax_demo | tee Source/04.00/softmax/softmax_results.txt


In [ ]:
from pathlib import Path

result_path = Path('Source/04.00/softmax/softmax_results.txt')
if not result_path.exists():
    raise FileNotFoundError('请先运行上一单元生成 softmax_results.txt')
lines = result_path.read_text(encoding='utf-8').splitlines()
header = 'shape\tavg_ms\tthroughput(MElts/s)\tmax_abs_error\tmax_row_sum_error\tstatus'
start = lines.index(header) + 1
rows = [line.split('\t') for line in lines[start:] if len(line.split('\t')) == 6]
columns = header.split('\t')
print('\t'.join(columns))
for row in rows: print('\t'.join(row))
assert rows and all(row[-1] == 'PASS' for row in rows)
print('所有 Softmax 测试均通过。')

## 6. 拓展实验

保持 `cols` 为 8 的倍数，分别改变 `rows`、`--blockdim`、`--repeat` 和 `--warmup`，观察行级并行、固定发射开销和统计稳定性。还可以把 `cols` 从 1024 改为 2048 或更大的 32 Byte 对齐长度，观察单行搬运和归约临时空间的变化。

建议每次只改变一个变量：固定 `rows` 比较不同 `blockdim`，固定 `blockdim` 比较不同 `rows`，再固定形状增加 `repeat`。这样才能把并行度、问题规模和计时噪声的影响区分开。

本版本的关键修复是编译边界：ASC 文件不再包含 `main`、`std::exp`、ACL 或 GE/Tiling Host 调用；这些代码只在 `host.cpp` 中由 CXX 编译并链接。
## 7. 实验总结

本实验把一个完整的 Softmax 算子拆成了可解释、可构建、可验证的几个层次：

- 数学层：使用减去行最大值的稳定公式，避免指数溢出；
- 数据结构层：用连续 GM 视图、行级 Local Tensor、输入/输出队列和归约缓冲组织数据生命周期；
- 并行层：利用行之间的独立性进行 AI Core 分配，同时保留行内归约；
- 工程层：将 ASC Kernel、CXX Host、共享配置和 CMake 链接边界明确分开；
- 验证层：同时检查逐元素最大绝对误差和每行输出和误差，并把状态写入可解析的汇总表。

如果三组规模都输出 `PASS`，说明 Kernel 的数值结果与 CPU 参考实现一致；随着规模扩大，平均时延和吞吐量的变化则可以帮助我们判断固定发射开销、访存量和设备并行利用率之间的关系。最重要的工程结论是：Ascend C 的 Kernel 文件不是普通的 C++ 应用入口，Host 代码和 Kernel 代码必须在正确的编译域中组织。

In [ ]:
extension_cases = [
    '--single --rows 8 --cols 1024 --blockdim 8',
    '--single --rows 16 --cols 1024 --blockdim 4',
    '--single --rows 16 --cols 2048 --blockdim 8',
]
for args in extension_cases:
    print(f'./Source/04.00/softmax/build/softmax_demo {args} --warmup 20 --repeat 200')